[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# How to Track Objects on a Moving Camera with BoT-SORT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roboflow/trackers/blob/develop/docs/cookbooks/how-to-track-a-moving-camera-with-botsort.ipynb)

A tracker predicts where each object will be in the next frame and matches the new detections to those predictions. When the camera moves, the whole image shifts, so a prediction made from the last frame lands far from the object and the object gets a new ID.

BoT-SORT adds camera motion compensation (CMC): before matching, it measures how the background moved between the two frames and moves every prediction by the same amount.

This notebook detects people in MOT17-13, filmed from a moving bus, with RF-DETR, tracks them twice with BoT-SORT, CMC off and on, and compares both runs.

## Setup

### Check GPU availability

Let's make sure that we have access to a GPU. We can use the `nvidia-smi` command to do that. If no GPU appears, navigate to `Runtime` → `Change runtime type` → `Hardware accelerator`, select a GPU, and click `Save`. The notebook also runs on a CPU runtime, only slower.

In [ ]:
!nvidia-smi

### Install dependencies

Dependency conflict warnings are expected and can be ignored.

In [ ]:
!pip install -q trackers==2.6.1 rfdetr

### Download MOT17

The validation split holds the second half of each MOT17 training sequence.

In [ ]:
from pathlib import Path

ROOT = Path("/content/mot17")

!trackers download mot17 --split val --asset frames --output {ROOT}

## Track with CMC off and on

RF-DETR detects people once per image, and both trackers get the same boxes, so any difference comes from the camera step. Pass the frame to `update()`: CMC measures the camera motion from it, and without a frame BoT-SORT skips that step. BoT-SORT marks tracks it has not confirmed yet with `tracker_id` `-1`, so drop those before drawing.

Both trackers process images from the start of the sequence, but the video keeps only images 105 to 190, where the bus turns into a side street. By the first shown image, each has assigned IDs 0 to 18, so both counters start at 19. The video is cropped to the right side of the frame and slowed to 8 FPS.

In [ ]:
import cv2
import numpy as np
import supervision as sv
from rfdetr import RFDETRLarge

from trackers import BoTSORTTracker

IMAGES = sorted((ROOT / "mot17" / "val" / "MOT17-13-FRCNN" / "img1").glob("*.jpg"))
FIRST_IMAGE, LAST_IMAGE = 105, 190
X0, Y0, X1, Y1 = 864, 216, 1920, 918
TARGET_VIDEO_PATH = "/content/mot17-13-cmc.mp4"

model = RFDETRLarge()
trackers = {
    "CMC off": BoTSORTTracker(frame_rate=25, enable_cmc=False),
    "CMC on": BoTSORTTracker(frame_rate=25, enable_cmc=True),
}
ids_so_far = {name: set() for name in trackers}

box_annotator = sv.BoxAnnotator(color_lookup=sv.ColorLookup.TRACK, thickness=3)
label_annotator = sv.LabelAnnotator(color_lookup=sv.ColorLookup.TRACK, text_color=sv.Color.BLACK, text_scale=0.8)

with sv.VideoSink(TARGET_VIDEO_PATH, sv.VideoInfo(width=X1 - X0, height=2 * (Y1 - Y0), fps=8)) as sink:
    for frame_index, image_path in enumerate(IMAGES[:LAST_IMAGE], start=1):
        frame = cv2.imread(str(image_path))

        # Detect people once; both trackers get the same boxes.
        detections = model.predict(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), threshold=0.3).with_nms(threshold=0.3)
        detections = detections[detections.data["class_name"] == "person"]

        panels = []
        for name, tracker in trackers.items():
            # Track. Pass frame for CMC
            tracked = tracker.update(detections, frame=frame)
            tracked = tracked[tracked.tracker_id != -1]
            ids_so_far[name].update(tracked.tracker_id.tolist())

            # Draw boxes and IDs, crop to see the relevant part and add the ID counter.
            labels = [f"#{tracker_id}" for tracker_id in tracked.tracker_id]
            panel = box_annotator.annotate(frame.copy(), tracked)
            panel = label_annotator.annotate(panel, tracked, labels=labels)[Y0:Y1, X0:X1].copy()
            cv2.rectangle(panel, (0, 0), (X1 - X0, 70), (0, 0, 0), -1)
            cv2.putText(panel, f"{name}   IDs so far: {len(ids_so_far[name])}", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (255, 255, 255), 3)
            panels.append(panel)

        # Write: keep only the window, CMC off on top.
        if frame_index >= FIRST_IMAGE:
            sink.write_frame(np.vstack(panels))

### Display result

In [ ]:
TARGET_VIDEO_COMPRESSED_PATH = "/content/mot17-13-cmc-compressed.mp4"

!ffmpeg -y -loglevel error -i {TARGET_VIDEO_PATH} -vcodec libx264 -crf 28 {TARGET_VIDEO_COMPRESSED_PATH}

In [ ]:
from IPython.display import Video

Video(TARGET_VIDEO_COMPRESSED_PATH, embed=True, width=600)

Watch the counters while the camera moves. From 19, the CMC-off counter rises to 33 (14 new IDs), while CMC on rises to 26 (7 new IDs). The man walking under the yellow signs keeps #24 with CMC on; with CMC off he gets seven IDs in ten images, #23 to #29.

You just tracked a moving camera with BoT-SORT and saw what camera motion compensation changes. Nice work!

CMC is on by default in `BoTSORTTracker`, so in your own code the one thing to remember is to pass the frame to `update()`. The [BoT-SORT page](https://trackers.roboflow.com/2.6.1/trackers/botsort/) covers its parameters.

Got feedback or ideas? Open an issue on [GitHub Issues](https://github.com/roboflow/trackers/issues).